In [ ]:
# Mount Google Drive
from google.colab import drive
import os

# Disable Weights & Biases tracking
os.environ["WANDB_DISABLED"] = "true"

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define directories and file paths
main_dir = "/content/drive/MyDrive/ITCS6156_BERT"
result_dir = os.path.join(main_dir, "Results")
data_dir = os.path.join(main_dir, "Data")
train_path = os.path.join(data_dir, "train_data.txt")
test_path = os.path.join(data_dir, "test_data.txt")
test_solution_path = os.path.join(data_dir, "test_data_solution.txt")


In [ ]:
import pandas as pd

# Load training and test data
train_data = pd.read_csv(train_path, sep=":::", engine="python", header=None, names=["ID", "TITLE", "GENRE", "DESCRIPTION"])
test_data = pd.read_csv(test_path, sep=":::", engine="python", header=None, names=["ID", "TITLE", "DESCRIPTION"])
test_solution = pd.read_csv(test_solution_path, sep=":::", engine="python", header=None, names=["ID", "TITLE", "GENRE", "DESCRIPTION"])

# Combine title and description for text input
train_data["TEXT"] = train_data["TITLE"] + " " + train_data["DESCRIPTION"]
test_data["TEXT"] = test_data["TITLE"] + " " + test_data["DESCRIPTION"]

# Encode genres to numeric labels
train_data["GENRE_LABEL"] = train_data["GENRE"].factorize()[0]
genre_mapping = dict(enumerate(train_data["GENRE"].factorize()[1]))

print("Genre Mapping:", genre_mapping)


In [ ]:
from sklearn.model_selection import train_test_split

# Split training data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_data["TEXT"], train_data["GENRE_LABEL"], test_size=0.2, random_state=42
)


In [ ]:
from transformers import DistilBertTokenizer

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize text data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(list(test_data["TEXT"]), truncation=True, padding=True, max_length=512)


In [ ]:
import tensorflow as tf

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(16)


In [ ]:
from transformers import TFDistilBertForSequenceClassification

# Load pre-trained DistilBERT model for sequence classification
num_labels = len(genre_mapping)
model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=7e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)


In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3  # Adjust based on dataset size
)


In [ ]:
# Evaluate the model
results = model.evaluate(val_dataset)
print("Validation Loss and Accuracy:", results)


In [ ]:
# Convert test_encodings to TensorFlow-compatible format
test_dataset = tf.data.Dataset.from_tensor_slices(dict(test_encodings)).batch(16)

# Predict on test data
test_predictions = model.predict(test_dataset)
test_predicted_labels = tf.argmax(test_predictions.logits, axis=1).numpy()

# Clean up genre mapping
clean_genre_mapping = {k: v.strip() for k, v in genre_mapping.items()}

# Map numeric predictions to genre names
test_data["PREDICTED_GENRE_LABEL"] = test_predicted_labels
test_data["PREDICTED_GENRE"] = test_data["PREDICTED_GENRE_LABEL"].map(clean_genre_mapping)

# Debugging output
print("Predicted Labels (first 20):", test_predicted_labels[:20])
print("Mapped Genres (first 20):", test_data["PREDICTED_GENRE"].head(20))

# Save predictions to CSV
predictions_file = os.path.join(result_dir, "predictions.csv")
test_data[["ID", "TITLE", "PREDICTED_GENRE"]].to_csv(predictions_file, index=False)
print(f"Predictions saved to {predictions_file}")


In [ ]:
# Inspect predicted labels
print("Predicted labels (first 20):", test_predicted_labels[:20])

# Print genre mapping
print("Genre Mapping:", genre_mapping)


In [ ]:
# Save the trained model and tokenizer
model.save_pretrained(os.path.join(result_dir, "distilbert_genre_model"))
tokenizer.save_pretrained(os.path.join(result_dir, "distilbert_genre_model"))
